In [2]:
import pandas as pd
import re
import pymorphy3
from tqdm import tqdm
from functools import lru_cache

morph = pymorphy3.MorphAnalyzer()
df = pd.read_csv('Data\\filtered_vacancies.csv', index_col = False)

In [2]:
morph.parse('деньгами')[0].normal_form

'деньга'

In [2]:
len(df)

10205

# 1) Rule-Based Tokenization

Пайплайн для токенизации: 
1) Привести текст к нижнему регистру
2) Заменить ё на е
3) Заменить пунктуацию на пустые строки
4) Заменить мусорные теги на пустые строки
5) Удалить лишние пробелы
6) Все слова (= НЕ теги) привести к стандартной(?) форме
7) Разбить текст на строки по ключевым тегам
8) Итерироваться по полученным строкам. Для каждой строки проверять, содержит ли она ключевое слово. Если содержит, то добавить все слова строки в соответствующий список токенов. Все дальнейшие строки добавлять в тот же список до тех пор, пока в очередной строке не встретиться другое ключевое слово. В этом случае изменить список, в который будут добавлять токены, и продолжить процесс. 

In [3]:
punctuation = set()
for text in df.Description:
    new_symbols = set(re.findall(r'[^a-zA-Zа-яА-Я0-9]', text))
    punctuation |= new_symbols

NECESSARY_PUNCTUATION = ['/', '.', '-', ':', '%', '+', '#', '$']
RUBBISH_PUNCTUATION = punctuation - set(NECESSARY_PUNCTUATION)

PUNCTUATION_TO_DELETE_ON_THE_BOUNDARIES = ['/', '-', ':']

MINUSES = ['–', '─', '―', '—', '‐', '‑', '‒', '⎯']

RUBBISH_TAGS = ['<b>', '</b>', '<strong>', '</strong>', '<i>', '</i>', '<em>', '</em>', '<u>', '</u>',
                '<mark>', '</mark>', '<span>', '</span>', '<hr />', '<link />', '<div title="Change Color">',
                '<div title="Copy">', '<div title="Delete">']
TAGS_TO_SAVE = ['h2', 'br /', 'ul', 'ol', '/h2', 'div', 'li', '/p', 'p', '/li', 
                '/ol', 'h4', 'h3', '/h3', '/div', '/ul', '/h4']

REQUIREMENTS_KEY_WORDS = ["искать", "требование", "пожелание", "нужный", "требоваться",
                           "ждать", "ожидать", "ожидание", "хотеть", "важно", 'необходимый',
                           'необходимо', 'нужно', "нужный", "обязательный", "обязательно", 
                           "желательно", "знание", "опыт", "умение", "навык", "понимание",
                           "владение", "экспертиза", "квалификация", "компетенция", "готовность", 
                           "способность", "рассматривать", "подходить", "кандидат", "соискатель", 
                           "плюс", "преимущество"]
RESPONSIBILITIES_KEY_WORDS = ["быть", "обязанность", "задача", "делать", "заниматься", 
                              "предстоять", "функционал", "зона", "ответственность", "функция",
                              "разработка", "поддержка", "сопровождение", "создание", "участие",
                              "реализация", "выполнение", "ведение", "организация", "обеспечение",
                              "контроль", "анализ", "проектирование", "взаимодействие", "координация",
                              "управление", "подготовка", "работа", "решение", "внедрение"]
CONDITIONS_KEY_WORDS = ["предлагать", "условие", "предложение", "получить", "предоставлять", "обеспечивать", 
                        "гарантировать", "оформление", "зарплата", "вознаграждение", "компенсация",
                        "бонус", "премия", "дмс", "страхование", "отпуск", "график", "удалёнка",
                        "удалённый", "гибридный", "офис", "обучение", "развитие", "рост", "карьера",
                        "льгота", "бенефит", "корпоративный"]

In [5]:
', '.join(list(RUBBISH_PUNCTUATION))

'😉, ⏺, –, Ё, ×, 📞, …, ”, ◦, ⠀, ✉, 🌐, ⎯, Ə, ❗, !, 🌍, 🔎, 🌱, ş, \u200f, 🔸, �, (, 🤩, 📱, 📝, 🛒, ң, ❓, 🙌, 🔝, 🖥, Ҳ, Â, ✤, \u200b, ĸ, \u200e, ❤, ️, 🌼,  , `, ☕, Š, ̈, ›, Ρ, ö, ⁃, É, ○, _, ─, ⃣, ү, 📢, ⸻, 🔹, ü, \u2060, ³, \u2028, ⏱, ❯, ®, —, і, \\, *, „, Ç, İ, é, 💗, 📊, 🔬, @, \u2002, ‒, 🔧, >, ☉, 💬, 📍, −, 💚, ⅓, », 🥇, ҳ, 🙂, №, 🛠, 🟢, ♥, è, \u202f, ¹, ◎, ↘, 🎓, ‑, [, 📉, 📜, 💸, ], Ɵ, 🏢, ±, Ü, ☸, 🎉, Ν, \uf0b7, 🌸, \u2063, 🌴, \u3000, |, 📈, ▽, 🛋, ✈, 🐾, І, ө, 📄, 🎄, 🎵, Α, 🏥, €, 📩, ç, ó, \u200d, 🍽, ,, ұ, µ, ~, ı, 📨, ‼, 📌, ·, ☆, ^, ², \u2003, 🎨, ⚠, Ү, 💻, ğ, 🌿, ✨, 🕘, 👥, 👋, ☎, 🌝, ❔, Қ, ⬇, ≤, 🤍, 🌟, 🛡, 🔄, 😊, →, ⭐, ‘, 🧩, ⛔, ⏰, ), ё, 🎁, ⚡, ✓, ⟶, Č, ✔, 👌, 🏆, ≈, ⚫, ə, ▎, 💎, °, ’, ∙, ў, 🕓, 🦸, 💯, 🙃, 🧱, ", 📑, ㅤ, ;, 🔥, χ, ₽, ≥, ․, «, 🚢, 💼, ™, ⇒, “, ↔, ✅, ̆, 📚, \U0010058a, 📅, =, \ufeff, 💫, ➡, 🎯, 🎶, ●, \U00100601, ‚, \U00100589, ☺, 🔘, 🔮, ¸, ❮, ә, 😎, 😄, ?, ▪, 🏗, 🥉, 💡, ⬥, 🚀, ✏, 💛, ❌, \uf0a7, •, 🔍, ➕, ―, 🤝, 🕑, 😌, 🔶, ‐, <, 🧠, 👉, Ў, \xad, ▸, ₸, 💰, қ, ғ, 💐, &, ⚙'

In [5]:
def rough_cleaning(text):

    cleared_text = text.replace('ё', 'е')
    cleared_text = cleared_text.replace('➕', '+')
    cleared_text = cleared_text.replace('․', '.')
    for minus in MINUSES:
        cleared_text = cleared_text.replace(minus, '-')

    for tag in RUBBISH_TAGS:
        cleared_text = re.sub(tag, ' ', cleared_text)
    
    for char in RUBBISH_PUNCTUATION:
        escaped_char = re.escape(char)
        cleared_text = re.sub(escaped_char, ' ', cleared_text)

    for char in PUNCTUATION_TO_DELETE_ON_THE_BOUNDARIES:
        escaped_char = re.escape(char)
        cleared_text = re.sub(rf'(\w+){escaped_char}(?=\s|$)', r'\1 ', cleared_text)
        cleared_text = re.sub(rf'(\s|^){escaped_char}(\w+)', r' \2', cleared_text)
        cleared_text = re.sub(rf'\s{escaped_char}\s', r' ', cleared_text)
    
    for tag in TAGS_TO_SAVE:
        cleared_text = re.sub(rf'(\s|^){tag}(\s|$)', f' <{tag}> ', cleared_text)

    cleared_text = re.sub(r'\s+', ' ', cleared_text)
    cleared_text = cleared_text.lower().strip()

    return cleared_text


@lru_cache(maxsize=200000)
def normalize_word(word):
    return morph.parse(word)[0].normal_form

def normalize_sentence(sentence):
    words = sentence.strip().split()
    return ' '.join(normalize_word(word) for word in words)

def classify_sentence(sentence):
    sent_types = [0, 0, 0]
    words = set(sentence.split(' '))
    for key in REQUIREMENTS_KEY_WORDS:
        if key in words:
            sent_types[0] = 1
            break
    for key in RESPONSIBILITIES_KEY_WORDS:
        if key in words:
            sent_types[1] = 1
            break
    for key in CONDITIONS_KEY_WORDS:
        if key in words:
            sent_types[2] = 1
            break
    is_change = int(sum(sent_types) > 0)
    return is_change, sent_types

In [26]:
rows = []

no_req, no_res, no_con = 0, 0, 0
n_omissions = 0
n_good_descriptions = 0

for i in tqdm(range(len(df))):

    id = df.loc[i, 'Id']
    description = df.loc[i, 'Description']
    

    req, res, con = [], [], []

    cleaned_description = rough_cleaning(description)
    sentences = re.findall(r'([^<]*)<[\w/]+>', cleaned_description)
    sentence_types = [0, 0, 0]

    for sentence in sentences:
        normalized_sentence = normalize_sentence(sentence)
        is_change, new_types = classify_sentence(normalized_sentence)
        if is_change:
            sentence_types = new_types
        if sentence_types[0] == 1:
            req.extend(
                token for token in normalized_sentence.split(' ')
                if token
                )
        if sentence_types[1] == 1:
            res.extend(
                token for token in normalized_sentence.split(' ')
                if token
                )
        if sentence_types[2] == 1:
            con.extend(
                token for token in normalized_sentence.split(' ')
                if token
                )
    
    req = list(set(req))
    res = list(set(res))
    con = list(set(con))

    if len(req) * len(res) * len(con) == 0:
        no_req += len(req) == 0
        no_res += len(res) == 0
        no_con += len(con) == 0
        n_omissions += 1
        continue
    else:
        n_good_descriptions += 1
        rows.append([id, req, res, con])

print(f'''
Descriptions without missings: {n_good_descriptions} ✔️
Descriptions with missings: {n_omissions} ❌
Missing requirements: {no_req}
Missing responsibilities: {no_res}
Missing conditions: {no_con}

Successfully processed descriptions: {round(n_good_descriptions / len(df) * 100, 2)}%
''')

resulting_table = pd.DataFrame(rows, columns=['Id', 'Requirements', 'Responsibilities', 'Conditions'])
resulting_table.to_csv('RRS.csv', index = False)

100%|██████████| 10205/10205 [01:27<00:00, 116.84it/s]



Descriptions without missings: 9718 ✔️
Descriptions with missings: 487 ❌
Missing requirements: 194
Missing responsibilities: 173
Missing conditions: 462

Successfully processed descriptions: 95.23%



In [6]:
df.head()

,Id,Name,Description,Professional_Role_Id,Professional_Role
0,129039824,Специалист технической поддержки,<p>Ищем специалистов <strong>БЕЗ ОПЫТА</strong...,121,Специалист технической поддержки
1,129043584,3D-визуализатор,<strong>Обязанности:</strong> <ul> <li>Подгото...,34,"Дизайнер, художник"
2,128944004,Главный разработчик Python,<p><strong>Кто нам нужен:</strong><br />Опытны...,96,"Программист, разработчик"
3,128021606,GenAI Product Analyst,<p><strong>ORO</strong> — Агентство маркетинго...,150,Бизнес-аналитик
4,128984804,Системный аналитик (ML),<p>В Центр развития Департамента информационны...,148,Системный аналитик


In [11]:
normalize_sentence(rough_cleaning(df[df.Id == 128079676].Description.loc[11]))

'<p> наш отдел unity-разработка это команда профессионал отвечать за создание и развитие standoff 2. мы разрабатывать и внедрять новый фича улучшать существующий функционал оптимизировать игровой процесс и графику. в наш задача добавление уникальный игровой механик проработка пользовательский опыт и постоянный обновление контент чтобы игра оставаться интересный и актуальной. <p> <p> <p> <p> задача который ждать вы <p> <ul> <li> оптимизация производительность игровой проект и внутренний инструмент <li> <li> анализ улучшение и автоматизация процесс разработка <li> <li> разработка и поддержка ci/cd-практика интеграция сторонний сервис и инструмент <li> <li> создание и развитие внутренний решение для автоматизация ускорение пайплайна и улучшение качество продукт <li> <li> разработка утилит и расширение для unity повышать продуктивность команда <li> <li> участие в формирование архитектура технологический решение внутри tech dev-команда <li> <ul> <p> какой навык пригодиться <p> <ul> <li> глу